# 🔱 VoiceBatch Studio v2.3.0 - [Chatterbox Turbo Update]
अब इसमें दुनिया का सबसे तेज़ और इमोशनल TTS मॉडल शामिल है।

In [ ]:
# @title 💤 Step 1: इंस्टॉलेशन (Chatterbox + XTTS)
import os
from IPython.display import display, Javascript
display(Javascript('function ClickConnect(){document.querySelector("colab-connect-button").click()}setInterval(ClickConnect,60000)'))

print("⏳ दोनों मॉडल्स लोड हो रहे हैं, इसमें 2-3 मिनट लग सकते हैं...")
!pip install -q gradio librosa soundfile coqui-tts torchcodec chatterbox-tts torchaudio
os.makedirs("outputs", exist_ok=True)
print("✅ सब कुछ तैयार है!")

In [ ]:
# @title 🚀 Step 2: ऐप लॉन्च करें (Dual Engine Mode)
app_code = r'''
import gradio as gr
import torch, librosa, os, re, numpy as np, soundfile as sf
from TTS.api import TTS
from chatterbox.tts_turbo import ChatterboxTurboTTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# लोड हो रहे हैं दोनों मॉडल्स
xtts_model = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
chatter_model = ChatterboxTurboTTS.from_pretrained(device=device)

def strict_hindi_filter(text):
    pattern = re.compile(r'[^\u0900-\u097F\s।,?!:;0-9\[\]]')
    return pattern.sub('', text)

def studio_pro_engine(engine_type, text, audio_sample, speed, pitch, lang, sil_rem):
    if not audio_sample: return None
    if lang == 'hi': text = strict_hindi_filter(text)
    
    out_path = 'outputs/VoiceBatch_Studio_Output.wav'
    
    if engine_type == 'XTTS v2 (Stable)':
        xtts_model.tts_to_file(text=text, speaker_wav=audio_sample, language=lang, file_path=out_path, split_sentences=True)
        y, sr = librosa.load(out_path)
    else:
        # Chatterbox Turbo Engine
        wav = chatter_model.generate(text, audio_prompt_path=audio_sample)
        sf.write(out_path, wav.cpu().numpy().squeeze(), chatter_model.sr)
        y, sr = librosa.load(out_path)
        
    # पोस्ट प्रोसेसिंग
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(out_path, y, sr)
    return out_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.3.0')
    with gr.Row():
        with gr.Column():
            engine = gr.Radio(['XTTS v2 (Stable)', 'Chatterbox Turbo (Fast)'], label='Select AI Engine', value='XTTS v2 (Stable)')
            txt = gr.Textbox(label='Script', lines=8, placeholder='Tip: Chatterbox में [laugh] या [chuckle] लिख सकते हैं।')
            smp = gr.Audio(label='Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr'], label='Language', value='hi')
            spd = gr.Slider(0.7, 1.4, 1.0, label="Speed")
            ptc = gr.Slider(-4, 4, 0, label="Pitch")
            sil = gr.Checkbox(label="Silence Remover", value=True)
            btn = gr.Button('Generate High Quality Audio 🔱', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Download Output')

    btn.click(studio_pro_engine, [engine, txt, smp, spd, ptc, lng, sil], out)
demo.launch(share=True, debug=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
!python app.py